# Alpha Analytics — Capstone Orchestrator

Thin notebook that drives the modules in `src/`. All heavy logic lives in the package; this notebook is for parameter tweaks, ad-hoc inspection, and visualisation.

Set your API keys in `.env` first (see `.env.example`).

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')

from src import config

## 1. Extract raw data
Prices (yfinance) → EPS surprises (FMP) → SEC XBRL financials with 8-K Item 2.02 tagging.

In [ ]:
from src.data import StockPriceExtractor, EpsExtractor, SecFinancialsExtractor

price_df = StockPriceExtractor(config.DATES.start, config.DATES.end).extract(
    config.TICKERS, save_csv=f'{config.DATA_DIR}/prices.csv'
)

eps_df = EpsExtractor(config.fmp_api_key()).extract(
    config.TICKERS, save_csv=f'{config.DATA_DIR}/eps_raw.csv'
)

sec_df = SecFinancialsExtractor(
    cik_map=config.CIK_MAP,
    concepts=config.SEC_CONCEPTS,
    user_agent=config.sec_user_agent(),
).extract(save_csv=f'{config.DATA_DIR}/sec_raw.csv')

## 2. Clean

In [ ]:
from src.cleaning import CleanDates, clean_sec_facts_df

price_df = CleanDates.clean(price_df)
eps_df   = CleanDates.clean(eps_df)
sec_df   = CleanDates.clean(sec_df)

sec_clean = clean_sec_facts_df(sec_df.reset_index(), fisc_end_map=config.FISCAL_END_MONTH)
sec_clean.to_csv(f'{config.DATA_DIR}/sec_financials_cleaned_true_q4.csv', index=False)
sec_clean.head()

## 3. Technical indicators

In [ ]:
from src.analysis import TechnicalAnalysis

ta = TechnicalAnalysis()
with_indicators = ta.add_indicators(price_df.reset_index())
ta.plot_indicators(with_indicators, ticker='AAPL')

## 4. Event windows

In [ ]:
from src.analysis import build_event_windows, plot_event_panel

event_df_10 = build_event_windows(eps_df, price_df, window=10, require_full_window=True)
event_df_15 = build_event_windows(eps_df, price_df, window=15, require_full_window=True)

import pandas as pd
event_df = pd.concat([event_df_10, event_df_15], ignore_index=True)
event_df_10.to_csv(f'{config.DATA_DIR}/eps_event_windows_10.csv', index=False)
event_df_15.to_csv(f'{config.DATA_DIR}/eps_event_windows_15.csv', index=False)

plot_event_panel(event_df, window=10, title_prefix='[W=10] ')

## 5. Cross-section: post-EPS returns × QoQ growth

In [ ]:
from src.analysis import (
    calculate_event_returns, calculate_qoq_growth,
    merge_returns_and_growth, analyze_correlations,
)

event_returns = calculate_event_returns(event_df_10)
qoq_growth    = calculate_qoq_growth(sec_clean)
merged_data   = merge_returns_and_growth(event_returns, qoq_growth)
clean_data, corr_matrix = analyze_correlations(merged_data)

## 6. Trading strategies

Four event-driven strategies with consistent per-event return frames.

In [ ]:
from src.strategies import (
    prepare_event_frame, summarise_strategy,
    backtest_post_earnings_momentum, backtest_contrarian,
    backtest_contrarian_agnostic, backtest_pre_earnings_runup,
    equity_curve,
)

MOMENTUM_H, CONTRARIAN_H, PRE_RUNUP_P, COST_BPS = 6, 1, 10, 0

df = prepare_event_frame(event_df_10)

mom = backtest_post_earnings_momentum(df, H=MOMENTUM_H, cost_bps=COST_BPS)
ctr = backtest_contrarian(df, H=CONTRARIAN_H, cost_bps=COST_BPS)
agn = backtest_contrarian_agnostic(df, H=CONTRARIAN_H, cost_bps=COST_BPS)
pre = backtest_pre_earnings_runup(df, P=PRE_RUNUP_P, cost_bps=COST_BPS)

summary = pd.DataFrame({
    f'Momentum +1..+{MOMENTUM_H}':     summarise_strategy(mom),
    f'Contrarian +1..+{CONTRARIAN_H}': summarise_strategy(ctr),
    f'Agnostic +1..+{CONTRARIAN_H}':   summarise_strategy(agn),
    f'Pre Run-up -{PRE_RUNUP_P}..-1':  summarise_strategy(pre),
}).T
summary

## 7. Pre-earnings strategy — research only

⚠️ The `surprise_filter` arg uses post-announcement information. See the **Look-Ahead Bias** section in the README. Anything quoted with a non-`None` filter is a research curiosity, not a tradable result.

In [ ]:
from src.strategies import PreEarningsStrategy

pe = PreEarningsStrategy(df=event_df_15)
tradable = pe.calculate_strategy_returns(entry_day=-10, exit_day=-1)
print('Tradable (no filter):', tradable['return'].mean(), (tradable['return'] > 0).mean())

biased = pe.calculate_strategy_returns(entry_day=-10, exit_day=-1, surprise_filter=0.05)
print('LOOK-AHEAD biased   :', biased['return'].mean(),  (biased['return'] > 0).mean())